In [81]:
using Combinatorics, ClusteredLowRankSolver, Infinity, LinearAlgebra, Nemo
include("fun.jl")

LDL (generic function with 1 method)

In [82]:
# [8,9,3], [10,5,4], [11,43,3], [14,296,3], [14,10,5], [15,139,4], [16,1171,3], [17,60,5], [18,922,4], [19,7421,3], [19,250,5]

n,K,d = [14,10,5]

R, x = polynomial_ring(QQ, :x)
N, z = number_field(x^2 - 3, :z)

name_file=string("cerf/general/$n","_$K","_$d.jl")

include(name_file) # includes Y-> set of positive matrices, C -> Vector dual variable

print("done")
# C[1]=C[1]-(2^n//K)*w 

done

In [ ]:
f4, size = f4_fun(n)
f2, size2 = f2_fun(n)

D = [sum([(2//1)^(-n)*K*Kw_field(j,i,n)*C[j+1] for j in 0:d-1]) for i in 0:n] # Define D

y=yijtp_fun_field(Y, n, f2, z) # Define yijtp

# Obj. funct

obj_fun = (D[1]-C[1]) - y[f4[0, 0, 0, 0]+1]/gamma_field(0,0,0,0,n)

println("Objective function = $obj_fun")
println("\n")

# Define Q

Q=[]

for i in 1:d-1
    qu=-D[i+1]+C[i+1]+(2*y[f4[i, 0, 0, 0]+1]+y[f4[i, i, i, i]+1])/gamma_field(i,0,0,0,n)
    
    push!(Q,qu/((2^n//K)-2))
end

for i in d:n
    qu=-D[i+1]+(2*y[f4[i, 0, 0, 0]+1]+y[f4[i, i, i, i]+1])/gamma_field(i,0,0,0,n)
    
    push!(Q,qu/((2^n//K)-2))
end

# Constraint
        
ref=0 

for (i, j, t, p) in keys(f4)
    if (t - p) % 2 == 0 && i != 0 && j != 0 && i+j-t-p !=0
    
        total=0
        
        for (a, b, c, q) in keys(f4) 
            
            if t - p == c - q  && [i, j, i + j - t - p] in collect(permutations((a, b, a + b - c - q)))
    
                total += y[f4[a, b, c, q]+1]/gamma_field(a,b,c,q,n)+ Q[a+b-c-q]
            end
        end
           
        ref += total
        
    end
end

println("Min value of the constraint = $ref")


# All constraints should be 0 and the objective function 1

Objective function = 1


Min value of the constraint = 0


In [84]:
# Maximum and minimum eigenvalue of Y

Ypsd=[]

for k in 1:size2
    dim=length(Y[k][1,:])
    
    M=Matrix(undef,dim,dim)
    
    for i in 1:dim
        for j in 1:dim
            
            if typeof(Y[k][i,j])==AbsSimpleNumFieldElem
                
                M[i,j]=BigFloat(coeff(Y[k][i,j],0))+BigFloat(coeff(Y[k][i,j],1))*sqrt(big(3))
            else
                M[i,j]=Y[k][i,j]
               
        
            end
        end
    end
    
    push!(Ypsd,M)
    
end

ref_min=Inf
ref_max=0
for i in 1:size2
    mat=Matrix{BigFloat}(Ypsd[i])
    
    eigen=eigvals(mat)

    min=minimum(eigen)
    max=maximum(eigen)

    if ref_min > min
        ref_min=min
    end
    if max > ref_max
        ref_max=max
    end
end

println("Minimum eigenvalue = $ref_min\n")
println("Maximum eigenvalue = $ref_max")

Minimum eigenvalue = 140218.7403580791228642566059471157755517885557479884330472187902132442642528965

Maximum eigenvalue = 9.789385626427501400406983483675758694281173296803545541876196832892485517741699e+14


In [85]:
## Check positivity with LDL method

ref_min=Inf

upper=rationalize(big"1.732050807568877293529")
lower=rationalize(big"1.732050807568877293528")

for k in 1:size2
    
    A=Y[k]
    D=length(A[1,:])
 
    d=LDL(A,D)
    # println(d)
    d_factor=Matrix(undef,D,1)
    
    for j in 1:D

        if coeff(d[j],1) < 0
            factor=coeff(d[j],0)+upper*coeff(d[j],1) # bigger value of sqrt(3)
        else
            factor=coeff(d[j],0)+lower*coeff(d[j],1) # smaller value of sqrt(3)
        end

        d_factor[j]=factor
    end
    
    ref=minimum(d_factor)

    if ref_min > ref
        ref_min=ref
    end
end


ref_min_float=BigFloat(ref_min)

println("Minimum value (Fraction) = $ref_min")
println("Minimum value (Float) = $ref_min_float")

Minimum value (Fraction) = 212524054332793051472619459703499087909879332230079151787145784019237552541690265826117757033209805807646200412071303416531429814941656211908754106119892062662565852404580383658868434799066535537195654937223520403986182509870955610337877540373837958304567320317297444019532858650857727896690127796876452205172185542834113848478772458805241584188119909213245694057196287884809761520573361622177737385721915277415254200611522513818160849236123216252872633302752603049318179357627727158965639727339759905446116634183066500641930108855484699465979898561045798043652622568114883258996577897728658810786775138560853465693922687738216645935685787543289403410577626849469194596605272103951763659857509239953758006224360172996170368981035566343443477182036024223425878699//19095990344269319992458488209462778352732132220312234379469822667623929228521731000371519960330786039859980670401673450262059153796473060933931310811403412600588547584309662006355684249499183241157870588025096